# Trabajo práctico: búsqueda voraz y A* sobre un grafo dirigido

**Estudiante:** Ezequiel Zunino
**Curso:** Inteligencia Artificial · semana 3, clase 3 · modalidad individual

El notebook implementa costo uniforme (UCS), búsqueda voraz y A* con un mismo esqueleto, y compara qué camino devuelve cada una, cuánto cuesta y cuántos estados expande. Corre de punta a punta solo con la biblioteca estándar de Python.

## 1. Representación del grafo y la heurística

El problema formal es `P=(S,A,T,s0,G,c)` con estados `{S,A,B,C,D,G}`, inicio `S`, objetivo `{G}`, transiciones dadas por las aristas dirigidas y costo `c` igual al valor de cada arista.

In [1]:
import heapq
from itertools import count

from IPython.display import Markdown, display

ESTADOS = {"S", "A", "B", "C", "D", "G"}

COSTOS = {
    ("S", "A"): 2,
    ("S", "B"): 2,
    ("A", "C"): 2,
    ("A", "D"): 5,
    ("B", "D"): 2,
    ("C", "G"): 3,
    ("D", "G"): 6,
}

H = {"S": 7, "A": 5, "B": 7, "C": 3, "D": 6, "G": 0}

INICIAL = "S"
OBJETIVO = "G"

## 2. Esquema común

Cada nodo conserva `(estado, padre, accion, g, h, f)`, donde `f` es la prioridad con la que entra a la frontera. Los tres algoritmos usan la misma función `buscar` y solo cambia cómo se calcula esa prioridad.

Decisiones de implementación:

- **Cola de prioridad:** `heapq` con entradas `(prioridad, orden, nodo)`. `orden` es un contador de inserción único, así que ante prioridades iguales sale primero el que se insertó antes (FIFO) y nunca se comparan dos nodos entre sí.
- **Prueba del objetivo:** se hace al extraer, no al generar.
- **Obsoletos:** una entrada cuyo `g` ya no es el mejor conocido se descarta al extraerla y no cuenta como expansión. El pseudocódigo de voraz no lo escribe, pero la sección 2 de la consigna lo pide en el esqueleto común, y acá se aplica a los tres.
- **Generados:** como en `A_estrella_base.py`, cuenta los nodos insertados en la frontera, incluida la raíz.
- **Frontera máxima:** cuenta las entradas físicas de la cola, incluidas las obsoletas que todavía no se extrajeron.

### Dos conteos de reaperturas

La consigna define reapertura como "veces que mejora `mejor_g` de un estado ya conocido". `A_estrella_base.py`, en cambio, solo la cuenta si ese estado ya había sido expandido. No son lo mismo, así que se informan los dos:

- `reaperturas`: definición de la consigna.
- `reaperturas_expandidos`: definición del código base.

In [2]:
def crear_nodo(estado, padre=None, accion=None, g=0, h=0, f=0):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "g": g,
        "h": h,
        "f": f,
    }


def sucesores(estado, costos):
    return [
        (destino, costo)
        for (origen, destino), costo in costos.items()
        if origen == estado
    ]


def reconstruir_camino(nodo):
    camino = []
    while nodo is not None:
        camino.append(nodo["estado"])
        nodo = nodo["padre"]
    camino.reverse()
    return camino


def vista_frontera(frontera):
    """Devuelve (estado, prioridad) en el orden en que saldrian de la cola."""
    return [(n["estado"], p) for p, _, n in sorted(frontera, key=lambda e: e[:2])]


def buscar(prioridad, h=H, costos=COSTOS, inicial=INICIAL, objetivo=OBJETIVO):
    """Busqueda de mejor primero; prioridad(g, h) define la estrategia."""
    orden = count()
    raiz = crear_nodo(inicial, g=0, h=h[inicial], f=prioridad(0, h[inicial]))
    frontera = [(raiz["f"], next(orden), raiz)]
    mejor_g = {inicial: 0}
    ya_expandidos = set()
    resultado = {
        "camino": None,
        "costo": None,
        "generados": 1,
        "expandidos": 0,
        "frontera_maxima": 1,
        "reaperturas": 0,
        "reaperturas_expandidos": 0,
        "traza": [],
    }

    while frontera:
        _, _, nodo = heapq.heappop(frontera)
        estado = nodo["estado"]
        paso = {
            "extraido": estado,
            "g": nodo["g"],
            "h": nodo["h"],
            "f": nodo["f"],
            "frontera": vista_frontera(frontera),
        }

        if nodo["g"] > mejor_g[estado]:
            resultado["traza"].append({**paso, "evento": "descartado (obsoleto)"})
            continue

        if estado == objetivo:
            resultado["traza"].append({**paso, "evento": "objetivo"})
            resultado["camino"] = reconstruir_camino(nodo)
            resultado["costo"] = nodo["g"]
            return resultado

        resultado["traza"].append({**paso, "evento": "expandido"})
        ya_expandidos.add(estado)
        resultado["expandidos"] += 1

        for siguiente, costo in sucesores(estado, costos):
            nuevo_g = nodo["g"] + costo
            if siguiente in mejor_g and nuevo_g >= mejor_g[siguiente]:
                continue
            if siguiente in mejor_g:
                resultado["reaperturas"] += 1
            if siguiente in ya_expandidos:
                resultado["reaperturas_expandidos"] += 1

            mejor_g[siguiente] = nuevo_g
            hijo = crear_nodo(
                siguiente,
                padre=nodo,
                accion=f"{estado} -> {siguiente}",
                g=nuevo_g,
                h=h[siguiente],
                f=prioridad(nuevo_g, h[siguiente]),
            )
            heapq.heappush(frontera, (hijo["f"], next(orden), hijo))
            resultado["generados"] += 1

        resultado["frontera_maxima"] = max(
            resultado["frontera_maxima"], len(frontera)
        )

    return resultado

## 3. Los tres algoritmos

| Algoritmo | Prioridad | Qué privilegia |
|---|---|---|
| UCS | `g` | el camino más barato recorrido |
| Voraz | `h` | el estado que parece más cerca del objetivo |
| A* | `g + h` | el costo total estimado de la solución |

In [3]:
def costo_uniforme(**opciones):
    return buscar(lambda g, h: g, **opciones)


def voraz(**opciones):
    return buscar(lambda g, h: h, **opciones)


def a_estrella(**opciones):
    return buscar(lambda g, h: g + h, **opciones)


ucs = costo_uniforme()
vor = voraz()
ast = a_estrella()

## 4. Traza de cada algoritmo

Cada fila es una extracción de la frontera. La columna `frontera` muestra lo que queda en la cola justo después de extraer y antes de agregar sucesores, con el formato `estado(prioridad)` y en orden de salida.

In [4]:
def tabla_traza(resultado):
    filas = [
        "| Paso | Extraído | g | h | Prioridad | Evento | Frontera restante |",
        "|---:|---|---:|---:|---:|---|---|",
    ]
    for n, paso in enumerate(resultado["traza"], start=1):
        frontera = " ".join(f"{e}({p})" for e, p in paso["frontera"]) or "(vacía)"
        filas.append(
            f"| {n} | {paso['extraido']} | {paso['g']} | {paso['h']} "
            f"| {paso['f']} | {paso['evento']} | {frontera} |"
        )
    return "\n".join(filas)


for nombre, resultado in [("UCS", ucs), ("Voraz", vor), ("A*", ast)]:
    display(Markdown(f"### {nombre}\n\n{tabla_traza(resultado)}"))

### UCS

| Paso | Extraído | g | h | Prioridad | Evento | Frontera restante |
|---:|---|---:|---:|---:|---|---|
| 1 | S | 0 | 7 | 0 | expandido | (vacía) |
| 2 | A | 2 | 5 | 2 | expandido | B(2) |
| 3 | B | 2 | 7 | 2 | expandido | C(4) D(7) |
| 4 | C | 4 | 3 | 4 | expandido | D(4) D(7) |
| 5 | D | 4 | 6 | 4 | expandido | D(7) G(7) |
| 6 | D | 7 | 6 | 7 | descartado (obsoleto) | G(7) |
| 7 | G | 7 | 0 | 7 | objetivo | (vacía) |

### Voraz

| Paso | Extraído | g | h | Prioridad | Evento | Frontera restante |
|---:|---|---:|---:|---:|---|---|
| 1 | S | 0 | 7 | 7 | expandido | (vacía) |
| 2 | A | 2 | 5 | 5 | expandido | B(7) |
| 3 | C | 4 | 3 | 3 | expandido | D(6) B(7) |
| 4 | G | 7 | 0 | 0 | objetivo | D(6) B(7) |

### A*

| Paso | Extraído | g | h | Prioridad | Evento | Frontera restante |
|---:|---|---:|---:|---:|---|---|
| 1 | S | 0 | 7 | 7 | expandido | (vacía) |
| 2 | A | 2 | 5 | 7 | expandido | B(9) |
| 3 | C | 4 | 3 | 7 | expandido | B(9) D(13) |
| 4 | G | 7 | 0 | 7 | objetivo | B(9) D(13) |

**Lectura de la traza de UCS.** D entra primero con `g=7` desde A y mejora a `g=4` cuando se expande B. La entrada vieja de D queda en la cola. En el paso 6 empata en prioridad 7 con G, sale antes por FIFO y se descarta por obsoleta. Por eso se ven seis extracciones antes de G, pero solo cinco expansiones.

## 5. Tabla comparativa

In [5]:
def fila(titulo, valor):
    return f"| {titulo} | " + " | ".join(str(valor(r)) for r in (ucs, vor, ast)) + " |"


tabla = [
    "| Resultado | UCS | Voraz | A* |",
    "|---|---|---|---|",
    fila("Camino", lambda r: " → ".join(r["camino"])),
    fila("Costo", lambda r: r["costo"]),
    "| Prioridad | `g` | `h` | `g+h` |",
    fila("Expandidos antes de extraer G", lambda r: r["expandidos"]),
    fila("Generados", lambda r: r["generados"]),
    fila("Frontera máxima", lambda r: r["frontera_maxima"]),
    fila("Reaperturas (consigna)", lambda r: r["reaperturas"]),
    fila("Reaperturas sobre expandidos (código base)", lambda r: r["reaperturas_expandidos"]),
]
display(Markdown("\n".join(tabla)))

| Resultado | UCS | Voraz | A* |
|---|---|---|---|
| Camino | S → A → C → G | S → A → C → G | S → A → C → G |
| Costo | 7 | 7 | 7 |
| Prioridad | `g` | `h` | `g+h` |
| Expandidos antes de extraer G | 5 | 3 | 3 |
| Generados | 7 | 6 | 6 |
| Frontera máxima | 3 | 3 | 3 |
| Reaperturas (consigna) | 1 | 0 | 0 |
| Reaperturas sobre expandidos (código base) | 0 | 0 | 0 |

## 6. Verificación

Estas comprobaciones fallan si cambia algún resultado que se usa en las respuestas de la sección 7. La heurística se chequea contra el costo real hasta G, calculado con UCS desde cada estado.

In [6]:
assert ucs["camino"] == vor["camino"] == ast["camino"] == ["S", "A", "C", "G"]
assert ucs["costo"] == vor["costo"] == ast["costo"] == 7
assert (ucs["expandidos"], vor["expandidos"], ast["expandidos"]) == (5, 3, 3)
assert (ucs["reaperturas"], vor["reaperturas"], ast["reaperturas"]) == (1, 0, 0)
assert ucs["reaperturas_expandidos"] == vor["reaperturas_expandidos"] == ast["reaperturas_expandidos"] == 0

# El objetivo se comprueba al extraer: UCS genera G y todavia expande D.
extraidos_ucs = [p["extraido"] for p in ucs["traza"]]
assert extraidos_ucs == ["S", "A", "B", "C", "D", "D", "G"]
assert [p["evento"] for p in ucs["traza"]].count("descartado (obsoleto)") == 1

# Costo real hasta G desde cada estado.
h_real = {e: costo_uniforme(inicial=e)["costo"] for e in ESTADOS}
admisible = all(H[e] <= h_real[e] for e in ESTADOS)
consistente = all(H[o] <= c + H[d] for (o, d), c in COSTOS.items())
assert admisible and consistente

print("Costo real hasta G:", dict(sorted(h_real.items())))
print("Heuristica:        ", dict(sorted(H.items())))
print("Admisible:", admisible, "| Consistente:", consistente)
print("Estados con h exacta:", sorted(e for e in ESTADOS if H[e] == h_real[e]))

Costo real hasta G: {'A': 5, 'B': 8, 'C': 3, 'D': 6, 'G': 0, 'S': 7}
Heuristica:         {'A': 5, 'B': 7, 'C': 3, 'D': 6, 'G': 0, 'S': 7}
Admisible: True | Consistente: True
Estados con h exacta: ['A', 'C', 'D', 'G', 'S']


### Comprobaciones para las preguntas 2 y 3

- **Pregunta 3:** A* con `h = 0` debe extraer los mismos estados, en el mismo orden, que UCS.
- **Pregunta 2:** un grafo chico muestra que la coincidencia de este ejercicio no es general. El camino `S→X→G` parece mejor porque `h(X)` es baja, pero es más caro.

In [7]:
h_cero = {e: 0 for e in ESTADOS}
ast_h0 = a_estrella(h=h_cero)
assert [p["extraido"] for p in ast_h0["traza"]] == extraidos_ucs
assert ast_h0["costo"] == ucs["costo"]
print("A* con h=0 extrae:", [p["extraido"] for p in ast_h0["traza"]])

otros_costos = {("S", "X"): 1, ("X", "G"): 10, ("S", "Y"): 2, ("Y", "G"): 2}
otra_h = {"S": 4, "X": 1, "Y": 2, "G": 0}
voraz_otro = voraz(h=otra_h, costos=otros_costos)
ast_otro = a_estrella(h=otra_h, costos=otros_costos)
assert voraz_otro["costo"] == 11 and ast_otro["costo"] == 4
print("Voraz:", voraz_otro["camino"], "costo", voraz_otro["costo"])
print("A*:   ", ast_otro["camino"], "costo", ast_otro["costo"])

A* con h=0 extrae: ['S', 'A', 'B', 'C', 'D', 'D', 'G']
Voraz: ['S', 'X', 'G'] costo 11
A*:    ['S', 'Y', 'G'] costo 4


## 7. Preguntas de análisis

**1. ¿Por qué voraz y A* coinciden en este grafo? ¿Qué condición del grafo y de la heurística lo explica?**

Coinciden porque la heurística es exacta en todo el camino óptimo: `h(S)=7`, `h(A)=5`, `h(C)=3` y `h(G)=0` son justo el costo real hasta G. Entonces, sobre ese camino, `f = g + h` se mantiene en 7, el costo óptimo. Además, en cada decisión el mejor estado según `h` es también el mejor según `g + h`: A le gana a B (`h` 5 contra 7, `f` 7 contra 9) y C le gana a D (`h` 3 contra 6, `f` 7 contra 13). La condición del grafo es que el camino que parece más cercano es también el más barato. El grafo no tiene una arista engañosa, con `h` baja y costo alto, que separe a los dos criterios. Por eso los dos expanden S, A y C y nunca abren B.

**2. ¿Garantiza voraz devolver el camino de menor costo en general?**

No. Su prioridad es solo `h(n)`: no mira `g`, así que no tiene en cuenta cuánto costó llegar a un nodo. Puede extraer el objetivo por un camino caro mientras queda en la frontera otro nodo que lleva a una solución mejor, porque G tiene `h = 0` y sale primero sin importar su `g`. A* no tiene ese problema: con `h` admisible, ningún nodo de un camino óptimo tiene `f` mayor que el costo óptimo, y por eso un objetivo más caro no puede salir antes. Voraz se queda con la rapidez, pero sin esa garantía. El grafo auxiliar de la sección 6 lo muestra: voraz devuelve costo 11 y A* devuelve costo 4.

**3. ¿Qué ocurre si se usa `h = 0` en A*? ¿Con qué algoritmo coincide entonces?**

Coincide con UCS: `f = g + 0 = g`, que es exactamente su prioridad. La verificación de la sección 6 lo confirma: extrae los mismos estados en el mismo orden. `h = 0` es admisible, así que el resultado sigue siendo óptimo, pero se pierde toda la orientación hacia el objetivo y A* vuelve a expandir 5 estados en lugar de 3.

**4. ¿Hubo reaperturas en UCS? ¿Y en A* y voraz? ¿Por qué?**

La respuesta depende de cuál de las dos definiciones se use:

- **Con la definición de la consigna** (mejorar `mejor_g` de un estado ya conocido), UCS tiene 1: D pasa de `g=7` por A a `g=4` por B. A* y voraz tienen 0, porque nunca expanden B y D solo se genera una vez, desde A.
- **Con la definición del código base** (mejorar un estado ya expandido), los tres tienen 0. En UCS la mejora de D ocurre antes de expandirlo.

Esto último no es casual. Con costos no negativos, UCS extrae cada estado por primera vez con su `g` óptimo. Lo mismo vale para A* cuando la heurística es consistente, y esta lo es. Voraz no tiene esa garantía: si tuviera 0 en otro grafo sería por la forma del grafo, no por una propiedad del algoritmo.

**5. ¿"Expandir menos estados" significa "camino más barato"?**

No. Son dos medidas distintas: la cantidad de expansiones mide el esfuerzo de búsqueda y el costo mide la calidad de la solución. Acá UCS expande 5 estados y voraz 3, y los dos devuelven el mismo camino de costo 7: voraz hizo menos trabajo, pero no encontró nada mejor. Tampoco se puede leer al revés. UCS expande más porque recorre todo lo que tiene costo acumulado menor que la solución, y eso es lo que le permite garantizar el óptimo. Voraz expande poco porque confía en `h`, y en el grafo auxiliar de la sección 6 esa confianza le cuesta una solución casi tres veces más cara. La igualdad de costos de este ejercicio se debe a que la heurística es muy buena en este grafo, no a que voraz sea óptimo.